In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification

In [11]:
news_filepath = f"../DATA-HTML-STOCK/NEPSENEWS/{symbol}news.csv"
save_filepath = f"../DATA-HTML-STOCK/STOCKSENTIMENT/{symbol}news_sentiment.csv"

query = "NEPSECompanyExtractor"
companyDetails = pd.read_csv(f"../DATA-HTML-STOCK/NEPSECompany/{query}.csv")
symbols = companyDetails["Symbol"]

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")
model = AutoModelForSequenceClassification.from_pretrained("ProsusAI/finbert")
model.eval()

In [13]:
def analyze_headline(headline):
    if not isinstance(headline, str) or headline.strip() in ["", "0"]:
        return None

    inputs = tokenizer(
        headline.strip(),
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=512
    )

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        probs = F.softmax(logits, dim=1)[0].numpy()

    predicted_class = int(np.argmax(probs))
    confidence = float(probs[predicted_class])

    label_map = {0: "negative", 1: "neutral", 2: "positive"}
    prediction = label_map[predicted_class]

    if prediction == "positive":
        sentiment_score = confidence
    elif prediction == "negative":
        sentiment_score = -confidence
    else:
        sentiment_score = 0.0

    return prediction, sentiment_score

In [14]:
news_df = pd.read_csv(news_filepath)

results = []

for _, row in news_df.iterrows():
    date = row.iloc[0] if len(row) > 0 else None
    headline = row.iloc[1] if len(row) > 1 else None

    analysis = analyze_headline(headline)

    if analysis:
        prediction, sentiment_score = analysis
        results.append({
            "Date": date,
            "Headline": headline,
            "Prediction": prediction,
            "Sentiment_Score": sentiment_score
        })

results_df = pd.DataFrame(results)

In [15]:
try:
    results_df["Date"] = pd.to_datetime(results_df["Date"])
    results_df = results_df.sort_values("Date", ascending=False)
except:
    pass

In [16]:
results_df.to_csv(save_filepath, index=False)

In [ ]:
print(f"{results_df}")